# Clinic Queue EDA (Smart Queue Dataset)

This notebook explores the clinic queue dataset, performs basic cleaning, builds requested features, and produces key visualizations.

Assumptions used for feature engineering:
- **queue_length** is defined as the number of appointments in the same `appointment_date`, `scheduled_hour`, and `department`.
- **hour_of_day** is taken directly from `scheduled_hour`.
- **doctor_avg_time** is the average `consultation_time` per `doctor_id`.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# Load dataset
from pathlib import Path

DATA_PATH = Path('smart_queue_dataset.csv')

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# Quick profile
print(df.dtypes)
print(df.isna().sum())
print('duplicates:', df.duplicated().sum())

In [ ]:
# Cleaning
clean = df.copy()

# Parse dates
clean['appointment_date'] = pd.to_datetime(clean['appointment_date'], errors='coerce')

# Coerce numeric columns
num_cols = [
    'patient_age', 'scheduled_hour', 'waiting_time_minutes', 'previous_no_shows',
    'queue_position', 'consultation_time', 'estimated_wait_time'
]
for col in num_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')

# Remove duplicate appointment_id if present
if 'appointment_id' in clean.columns:
    clean = clean.drop_duplicates(subset=['appointment_id'])
else:
    clean = clean.drop_duplicates()

# Fill missing values
for col in num_cols:
    if col in clean.columns:
        clean[col] = clean[col].fillna(clean[col].median())

cat_cols = clean.select_dtypes(include=['object']).columns
for col in cat_cols:
    clean[col] = clean[col].fillna('Unknown')

clean.isna().sum()

In [ ]:
# Feature engineering
clean['hour_of_day'] = clean['scheduled_hour']

# Queue length per date-hour-department
clean['queue_length'] = clean.groupby(
    ['appointment_date', 'scheduled_hour', 'department']
)['appointment_id'].transform('size')

# Doctor average consultation time
clean['doctor_avg_time'] = clean.groupby('doctor_id')['consultation_time'].transform('mean')

clean.head()

In [ ]:
# Patient load per day
patient_load = clean.groupby(clean['appointment_date'].dt.date)['appointment_id'].count().reset_index()
patient_load.columns = ['date', 'patient_count']
patient_load.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(patient_load['date'], patient_load['patient_count'], marker='o')
plt.title('Patient Load Per Day')
plt.xlabel('Date')
plt.ylabel('Patients')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Average consultation time
avg_consult = clean['consultation_time'].mean()
print('Average consultation time (minutes):', round(avg_consult, 2))

avg_consult_by_dept = clean.groupby('department')['consultation_time'].mean().sort_values(ascending=False)
avg_consult_by_dept

In [ ]:
# Department patient count
dept_counts = clean['department'].value_counts()
dept_counts

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=dept_counts.index, y=dept_counts.values)
plt.title('Department Patient Count')
plt.xlabel('Department')
plt.ylabel('Patients')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Histograms
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.histplot(clean['patient_age'], bins=20, kde=True, ax=axes[0])
axes[0].set_title('Patient Age')

sns.histplot(clean['waiting_time_minutes'], bins=20, kde=True, ax=axes[1])
axes[1].set_title('Waiting Time (min)')

sns.histplot(clean['consultation_time'], bins=20, kde=True, ax=axes[2])
axes[2].set_title('Consultation Time (min)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric features
num_df = clean.select_dtypes(include=[np.number])

plt.figure(figsize=(10, 6))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Patient flow graphs
# Appointments per hour
hourly = clean.groupby('hour_of_day')['appointment_id'].count().reset_index()

plt.figure(figsize=(8, 4))
plt.plot(hourly['hour_of_day'], hourly['appointment_id'], marker='o')
plt.title('Appointments by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Appointments')
plt.tight_layout()
plt.show()

# Average waiting time by hour
wait_by_hour = clean.groupby('hour_of_day')['waiting_time_minutes'].mean().reset_index()

plt.figure(figsize=(8, 4))
plt.plot(wait_by_hour['hour_of_day'], wait_by_hour['waiting_time_minutes'], marker='o', color='orange')
plt.title('Average Waiting Time by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Waiting Time (min)')
plt.tight_layout()
plt.show()

NameError: name 'clean' is not defined

In [ ]:
# Save clean dataset with engineered features
OUT_CLEAN = Path('smart_queue_dataset_clean.csv')
clean.to_csv(OUT_CLEAN, index=False)
print('Saved:', OUT_CLEAN.resolve())

In [ ]:
# Save feature documentation
feature_doc = '''
# Feature Documentation

## queue_length
Number of appointments scheduled in the same `appointment_date`, `scheduled_hour`, and `department`.

## hour_of_day
Hour of the appointment; copied from `scheduled_hour`.

## doctor_avg_time
Average `consultation_time` for each `doctor_id`.
'''.strip()

Path('feature_documentation.md').write_text(feature_doc, encoding='utf-8')
print('Feature documentation saved to', Path('feature_documentation.md').resolve())